# Lekcja 2 — Podstawy OFDM

## Cel nauki
- **Resource grid** — slot czasowo-częstotliwościowy (jak „obraz” dla CNN)
- **Subcarriers** — równoległe nośne w częstotliwości
- **OFDM symbols** — bloki IFFT/FFT w czasie
- **Piloty (DMRS)** — znane symbole do estymacji kanału

## Resource grid
```
       frequency →
time  [ D  D  P  D  D  P ]
      [ D  D  D  D  D  D ]
```
D = data, P = pilot

## Co zastąpi sieć?
CNN w lekcji 7 bierze **cały odebrany grid** `[Real, Imag, Time, Freq]` jako wejście. Musisz rozumieć ten tensor zanim zbudujesz sieć.

## Pipeline
```
Bits → QAM → ResourceGridMapper → OFDMModulator → Kanał → OFDMDemodulator → Grid RX
```


In [ ]:
import sys
from pathlib import Path

# Dodaj src/ do PYTHONPATH
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import sionna as sn
    import sionna.phy
except ImportError as e:
    raise ImportError(
        "Brak Sionny. Uruchom z katalogu magisterka/: ./scripts/drun sync"
    ) from e

from src.utils.setup import print_environment, get_device

sn.phy.config.seed = 42
device = get_device()
print_environment()


## Krok 1 — Definicja Resource Grid

In [ ]:
from sionna.phy.ofdm import ResourceGrid

rg = ResourceGrid(
    num_ofdm_symbols=14,
    fft_size=64,
    subcarrier_spacing=30e3,
    num_tx=1,
    num_streams_per_tx=1,
    cyclic_prefix_length=6,
    num_guard_carriers=[5, 6],
    dc_null=True,
    pilot_pattern="kronecker",
    pilot_ofdm_symbol_indices=[2, 11],
)
rg.show()
print("Liczba RE danych:", rg.num_data_symbols)
print("Liczba RE pilotów:", rg.num_pilot_symbols)


## Krok 2 — Mapowanie symboli QAM na grid

In [ ]:
from sionna.phy.ofdm import ResourceGridMapper
from sionna.phy.mapping import Mapper, Constellation

NUM_BITS_PER_SYMBOL = 4
constellation = Constellation("qam", NUM_BITS_PER_SYMBOL)
mapper = Mapper(constellation=constellation)
rg_mapper = ResourceGridMapper(rg)

batch_size = 4
num_data_symbols = rg.num_data_symbols
num_bits = num_data_symbols * NUM_BITS_PER_SYMBOL

bits = sn.phy.mapping.BinarySource()([batch_size, num_bits])
symbols = mapper(bits)
x_rg = rg_mapper(symbols)

print("Shape grid TX:", x_rg.shape)
# [batch, num_tx, num_streams, num_ofdm_symbols, fft_size]
print("Przykład |x| na RE danych:", torch.abs(x_rg[0, 0, 0]).mean().item())


## Krok 3 — Modulacja / demodulacja OFDM

In [ ]:
from sionna.phy.ofdm import OFDMModulator, OFDMDemodulator

modulator = OFDMModulator(rg.cyclic_prefix_length)
demodulator = OFDMDemodulator(rg.fft_size, 0, rg.cyclic_prefix_length)

x_time = modulator(x_rg)
print("Shape sygnał czasowy:", x_time.shape)

# Idealny kanał (brak zniekształceń) — tylko demodulacja
y_rg = demodulator(x_time)
print("Shape grid po demodulacji:", y_rg.shape)


## Krok 4 — Wizualizacja grida

In [ ]:
# Magnitude pierwszego batcha, pierwszy stream
mag = torch.abs(x_rg[0, 0, 0]).cpu().numpy()
plt.figure(figsize=(10, 4))
plt.imshow(mag, aspect="auto", origin="lower", cmap="viridis")
plt.colorbar(label="|x|")
plt.xlabel("Subcarrier")
plt.ylabel("OFDM symbol")
plt.title("Resource Grid — magnitude (TX)")
plt.show()


## Podsumowanie tensorów
| Etap | Shape | Typ |
|------|-------|-----|
| bits | `[B, num_bits]` | float 0/1 |
| symbols | `[B, 1, 1, num_data_RE]` | complex |
| grid | `[B, 1, 1, 14, 64]` | complex |
| time | `[B, 1, 1, num_samples]` | complex |

## Ćwiczenie
Porównaj `rg.num_data_symbols` z liczbą niezerowych elementów w gridzie.

**Następna lekcja:** `03_fading_channel.ipynb`
